In [ ]:
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATA_DIR = Path('../data')
IMAGE_DIR = DATA_DIR / "train_images"

train_df = pd.read_csv(DATA_DIR / "train.csv")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device", device)
print("데이터:", train_df.shape)

In [ ]:
# Multi-hot Encoing

classes = [
    "scab",
    "healthy",
    "frog_eye_leaf_spot",
    "complex",
    "rust",
    "powdery_mildew"
]

def multi_hot_encode(labels):
    label_list = labels.split()

    encoded = []
    for cls in classes:
        if cls in label_list:
            encoded.append(1)
        else:
            encoded.append(2)
    return encoded

In [ ]:
train_data = pd.read_csv(DATA_DIR/"train_split.csv")
valid_data = pd.read_csv(DATA_DIR/"valid_split.csv" )

print("Train:", len(train_data))
print("Valid:", len(valid_data))

In [ ]:

class PlanetDataset(Dataset):
    def __init__(self, df, image_dir, transform = None):
        self.df = df.reset_index(drop = True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx): # Dataset에서 이미지 한 장을 요청하면 해당 이미지와 정답을 반환하는 역할
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["image"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            multi_hot_encode(row["labels"]),
            dtype = torch.float32
        )

        return image, label

In [ ]:
# Transform 정의

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [ ]:
# Dataset과 DataLoader 생성

train_dataset = PlanetDataset(
    train_data,
    IMAGE_DIR,
    transform = transform
)

valid_dataset = PlanetDataset(
    valid_data,
    IMAGE_DIR,
    transform = transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 0
)

In [ ]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Labels dtype:", labels.dtype)

In [ ]:
# ResNet18로 학습

# ResNet18의 구조와 사전학습된 가중치
# 마지막 분류기를 Linear(512, 6)으로 변경하는 이유
# Freeze와 Xavier Initialization을 언제 사용하는가


# ResNet18 불러오기
# ImageNet 데이터셋의 1,000개의 클래스를 분류하도록 사전학습된 모델이기 때문에 출력이 1000개

import torch.nn as nn
from torchvision import models

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights = weights)

print(model.fc)

In [ ]:
model.fc = nn.Linear(512, 6)
print(model.fc)

In [ ]:
# Freeze란 무엇인가?
# 사전학습된 ResNet18은 이미 일반적인 이미지에서 유용한 시각적 특징을 학습한 상태
# 우리가 처음부터 모든 가중치를 변경할 필요가 없음
# 그래서 처음에는 기존 특징 추출기의 가중치를 고정, 새로 추가한 분류기만 학습


# 이렇게 하면 기존 ResNet18의 가중치는 Gradient를 계산하지 않고, 새로 생성한 fc의 가중치만 학습하게 됨
# requires_grad = False는 가중치 업데이트를 막는 것이지 BatchNorm의 Running Mean과 Variance까지 자동으로 고정하는 것은 아님
# 완전히 고정된 특징 추출기로 실험하려면 학습 시 BatchNorm 상태도 고려해야 됨


# Xavier Intialization은 지금 필요할까?
# model.Linear(512, 6)이 코드를 실행하면 PyTorch는 새로운 Linear Layer의 가중치를 기본 초기화 방식으로 자동 초기화
# 반드시 Xavier 초기화를 적용해야 하는 것은 아님


for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(512, 6)

In [ ]:
# 모델 출력 확인하기

model = model.to(device)
images = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

print("Output Shape:", outputs.shape)
print("Output Example:", outputs[0])

In [ ]:
# 모델 학습

import torch
import torch.nn as nn
from torchvision import models

torch.manual_seed(42)

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights = weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(
    model.fc.in_features,   # model.fc.parameters()를 사용하는 이유는 현재 기존 ResNet18의 특징 추출기를 Freeze했기 때문에
                            # 새로운 분류기의 가중치만 업데이트하면 되기 때문
                            # model.parameters()를 사용해도 requires_grad = False인 파라미터는 일반적으로 업데이트 되지 않지만, 학습 대상을 명확하게 표현하기 위해서 작성                
    6
)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr = 0.001,
    weight_decay=0.01
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):  # ResNet18에서 Batch Normalization 계측이 학습 중에 사용하는 통계값을 업데이트하지 않도록 설정하는 코드
            module.eval()

    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. Gradient 초기화
        optimizer.zero_grad()

        # 2. Forward
        outputs = model(images)
        
        # 3. Loss 계산
        loss = criterion(outputs, labels)

        # 4. Backward
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
# Validation 함수 구현

from sklearn.metrics import f1_score
import numpy as np

def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            preds = torch.sigmoid(outputs)

            total_loss += loss.item() * images.size(0)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(preds.cpu().numpy())

    valid_loss = total_loss / len(loader.dataset)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    valid_f1 = f1_score(
        all_labels,
        all_preds,
        average = "macro",
        zero_division=0
    )

    return valid_loss, valid_f1

In [ ]:
EPOCHS = 5

train_losses = []
valid_losses = []
valid_f1_scores = []

best_f1 = -1

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_f1 = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    valid_f1_scores.append(valid_f1)

    print(
        f"Epoch [{epoch+1} / {EPOCHS}]"
        f"Train Loss: {train_loss:.4f}"
        f"Valid Loss: {valid_loss:.4f}"
        f"Valid F1: {valid_f1:.4f}"
    )

    if valid_f1 > best_f1:
        best_f1 = valid_f1

        torch.save(
            model.state_dict(),
            "best_resnet18.pth"
        )

print(f"\nBest Valid Macro F1: {best_f1:.4f}")

In [ ]:
import time

start = time.perf_counter()

images, labels = next(iter(train_loader))

end = time.perf_counter()

print(f'DataLoader 시간: {end - start:.2f}초')
print('Images:', images.shape)

In [ ]:
images = images.to(device)

model.eval()

torch.cuda.synchronize()
start = time.perf_counter()

with torch.no_grad():
    outputs = model(images)

torch.cuda.synchronize()
end = time.perf_counter()

print(f"GPU Forward 시간: {end - start:.4f}초")